# The Cost of Discretion — Study v2 progress report

## A stronger way to study Formula 1 stewarding

The first report found a real limitation: formal FIA decisions alone do not contain enough common
detail to call a ruling consistent or inconsistent. Study v2 builds the missing structure. It adds
an independent-review packet, a public Race Control funnel, incident-lap windows, close-case
matching, and one harm record per driver in a collision.

The work below is reproducible, but it is not a finished fairness verdict. Human review is still
open, and the report keeps every result behind the relevant gate.

In [1]:
# ruff: noqa: E402
import json
import os
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".jupyter" / "mplconfig"))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

REVIEW = ROOT / "data/manual/study_v2_review_packets/study-v2-review-7cb1b29b5251"
REFERRAL = ROOT / "data/manual/study_v2_referrals/referrals-5d0559ad2878"
CLOCK = ROOT / "data/manual/study_v2_incident_clock/incident-clock-3dc8bb350308"
CONTEXT = ROOT / "data/manual/study_v2_incident_context/incident-context-a1de9ac2c8ea"
CLOSE = ROOT / "data/manual/study_v2_close_cases/close-cases-36f9bc70de82"
DAMAGE = ROOT / "data/manual/study_v2_damage/damage-screening-66381b550583"
LAYERS = ROOT / "data/manual/study_v2_layers/study-v2-layers-a9b8ff776470"
NATIONALITY = ROOT / "data/manual/study_v2_nationality/nationality-diagnostic-2b1b0ffdd961"
GENERATED = ROOT / "reports/generated/study_v2"
GENERATED.mkdir(parents=True, exist_ok=True)

In [2]:
review = json.loads((REVIEW / "manifest.json").read_text(encoding="utf-8"))
referral = json.loads((REFERRAL / "manifest.json").read_text(encoding="utf-8"))
clock = json.loads((CLOCK / "manifest.json").read_text(encoding="utf-8"))
close = json.loads((CLOSE / "manifest.json").read_text(encoding="utf-8"))
damage = json.loads((DAMAGE / "manifest.json").read_text(encoding="utf-8"))
layers = json.loads((LAYERS / "manifest.json").read_text(encoding="utf-8"))
nationality = json.loads((NATIONALITY / "manifest.json").read_text(encoding="utf-8"))

status = pd.DataFrame([
    {"part": "Independent source review", "built": f"{review['reviewer_a_rows']} A / {review['reviewer_b_rows']} B rows", "release": "Waiting for human review"},
    {"part": "Race Control referral links", "built": f"{referral['high_confidence_link_count']} high-confidence links", "release": "Descriptive"},
    {"part": "Incident clock mapping", "built": f"{clock['mapped_case_count']} of {clock['case_count']} cases", "release": "Validated candidate context"},
    {"part": "Close-case support", "built": f"{close['pre_review_minimum_support_count']} of {close['case_count']} cases", "release": "Review leads only"},
    {"part": "Collision harm records", "built": f"{damage['participant_record_count']} driver records", "release": "Screening only"},
    {"part": "Persistent pace", "built": f"{layers['pace_screen_estimable_rows']} estimable screens", "release": "Waiting for source/context review"},
    {"part": "Proportionality", "built": f"{layers['proportionality_release_rows']} release-ready rows", "release": "Withheld"},
    {"part": "Nationality", "built": f"{nationality['british_accused_rows']} British-accused cases", "release": "Inconclusive"},
])
display(status)

,part,built,release
0,Independent source review,496 A / 158 B rows,Waiting for human review
1,Race Control referral links,174 high-confidence links,Descriptive
2,Incident clock mapping,338 of 346 cases,Validated candidate context
3,Close-case support,317 of 346 cases,Review leads only
4,Collision harm records,411 driver records,Screening only
5,Persistent pace,28 estimable screens,Waiting for source/context review
6,Proportionality,0 release-ready rows,Withheld
7,Nationality,44 British-accused cases,Inconclusive


## 1. What became stronger

**The population boundary is more visible.** The public timing feed contains 966 Race Control
episodes. Of 346 formal primary decisions, 174 link to an episode at high confidence. Candidate and
ambiguous links remain visible rather than being forced into the analysis.

**Incident timing is much better.** FIA local incident clocks map 338 of 346 cases into lap windows.
The method reproduces all 31 cases that already had a known lap. It gives 174 single-lap candidates;
wider windows remain uncertain.

**Similar cases are compared without looking at the result.** The matching process excludes fault,
penalty, damage, retirement, and finish. It finds at least five neighbors for 317 cases. A different
outcome inside a close pair is a reason to read both sources, not a finding that either ruling was
wrong.

**Harm now follows every participant.** The 233 collision decision rows reduce to 193 candidate
incidents and expand to 411 driver-level harm records. This handles chain collisions and different
types of harm to different drivers.

## 2. Damage evidence and pace loss

No single public database reliably records Formula 1 damage. The collection method therefore joins
FIA timing and classifications with official team reports, named driver or engineer accounts, and
Formula1.com reporting. Team accounts can identify a floor, wing, puncture, repair, or attributed
pace cost, but they are interested-party evidence and must be checked against official timing.

Driver-specific clock mapping gives a single incident lap for 240 harm records. Fifty-two have the
minimum clean laps before and after plus teammate coverage. Exact same-lap matching leaves 28
estimable timing screens. These are not confirmed damage effects: tyre choice, traffic, strategy,
weather, and hidden car conditions can still drive the result.

The source method is documented with official examples, including
[Hamilton's attributed Imola front-wing loss](https://www.formula1.com/en/latest/article/front-wing-damage-cost-hamilton-0-6s-per-lap-until-imola-red-flag-mercedes.4YdB5ZdPJaoMCfjnx5Nk3u),
[Piastri's Miami wing change](https://www.formula1.com/en/latest/article/sainz-hit-with-five-second-time-penalty-after-collision-with-piastri-in.3D1JHk6lYz0GzKch77GcrZ), and
[Williams' description of worsening floor damage in Japan](https://www.williamsf1.com/posts/05f49fb5-62ac-4308-b14d-52f8959cfee8/2023-japanese-grand-prix).

In [3]:
display(Markdown("![Referral funnel](../reports/generated/study_v2/referral_funnel.png)"))
display(Markdown("![Pace screens](../reports/generated/study_v2/pace_screen_distribution.png)"))
display(Markdown("![Nationality power](../reports/generated/study_v2/nationality_power_v2.png)"))

![Referral funnel](../reports/generated/study_v2/referral_funnel.png)

![Pace screens](../reports/generated/study_v2/pace_screen_distribution.png)

![Nationality power](../reports/generated/study_v2/nationality_power_v2.png)

## 3. Conduct, harm, and punishment stay separate

Study v2 does not create one fairness score. It stores:

1. the act and the written finding;
2. each participant's observed consequence;
3. the nominal and realized cost of the sanction, in seconds, positions, grid places, or points.

A proportionality comparison requires independently reviewed fault, harm, and sanction application.
No full-corpus record meets every gate yet, so the release count is zero. This is an intended safety
feature, not a failed analysis.

The FIA's public 2025 guideline explanation says the guidelines assist steward decisions but are
not regulations. Historical FIA practice was also described as judging the incident rather than its
outcome. Damage therefore measures consequence; it does not back-fill fault.

## 4. Nationality remains inconclusive

British accused drivers received sanctions in 25 of 44 formal cases (56.8%). Other accused drivers
received sanctions in 189 of 302 cases (62.6%). This raw 5.8-point difference is not an adjusted
effect and does not show favoritism.

Measured overlap passes the frozen balance checks, but the British group is below the required 98
cases. Simulated power for the prespecified 15-point difference is only 37.8% to 53.6%, depending on
the baseline rate. Independent source review is unfinished. No adjusted nationality result is fit
or released.

## 5. What the reviewer needs to do next

The remaining work is judgment, not bulk data entry:

- complete the blinded Reviewer A and Reviewer B source packets;
- reconcile disagreements without showing either reviewer the model answer first;
- confirm the context fields for close pairs;
- review the highest-priority damage sources and any claimed repair, retirement, or benefit;
- code when and where a sanction was applied before calling nominal seconds an actual race cost.

After those gates pass, the same notebooks can release reviewed close-pair summaries, damage
coverage, participant harm, and proportionality comparisons. Until then, the defensible conclusion
is narrower: the study now has a much stronger design and a clear path to the answer, but not enough
independent evidence for a population-level fairness verdict.

## Evidence status

| Output | Current status |
|---|---|
| Model-reviewed 346-case population | Descriptive; disclosed model review |
| Referral funnel | Descriptive public-feed coverage |
| Incident lap windows | Validated candidate context |
| Close-case neighbors | Review-priority tool |
| Damage and pace screens | Source-research tool |
| Proportionality | Withheld pending independent review |
| Nationality effect | Inconclusive; release gate failed |

The protocol, source hierarchy, packets, transformations, and release gates are versioned in the
repository. Unknowns remain unknown instead of being converted into zeros.